In [3]:
!pip install -q albumentations==2.0.8 timm==1.0.28 opencv-python-headless pandas scikit-learn tqdm huggingface_hub requests

In [4]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import time
import random as py_random
import requests

from huggingface_hub import hf_hub_url, login, get_token

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

REPO_ID = "hunglc007/ThyroidXL"
REPO_TYPE = "dataset"
REPO_REVISION = "b15fe293bd74f1a8a4f05bf88bcdf06a1934125f"

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/ThyroidXL_Publication_OneFold")
MODEL_DIR = DRIVE_PROJECT_ROOT / "Models" / "ConvNeXtTiny"
RESULTS_DIR = DRIVE_PROJECT_ROOT / "results" / "ConvNeXtTiny"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = Path("/content/thyroidxl_train_cache")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)
    HF_TOKEN = get_token()

if not HF_TOKEN:
    raise RuntimeError("No Hugging Face token is available after login.")

AUTH_HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

def cached_remote_file(repo_path, max_attempts=20):
    repo_path = str(repo_path).replace("\\", "/").lstrip("/")


    if repo_path.startswith("test/"):
        raise RuntimeError(
            "TEST ACCESS BLOCKED in this training notebook. "
            f"Attempted path: {repo_path}"
        )

    destination = CACHE_ROOT / repo_path
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.is_file() and destination.stat().st_size > 0:
        return destination

    partial = destination.with_suffix(destination.suffix + ".part")
    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=repo_path,
        repo_type=REPO_TYPE,
        revision=REPO_REVISION,
    )

    for attempt in range(1, max_attempts + 1):
        try:
            with requests.get(
                url,
                headers=AUTH_HEADERS,
                stream=True,
                allow_redirects=True,
                timeout=(30, 300),
            ) as response:

                if response.status_code == 429:
                    retry_after = response.headers.get("Retry-After")
                    try:
                        wait = max(30, int(float(retry_after)))
                    except Exception:
                        wait = min(300, 30 * attempt)
                    print(
                        f"HTTP 429 for {repo_path}. "
                        f"Waiting {wait}s..."
                    )
                    time.sleep(wait + py_random.uniform(0, 5))
                    continue

                if response.status_code == 404:
                    raise FileNotFoundError(repo_path)

                if response.status_code in (401, 403):
                    raise PermissionError(
                        f"Hugging Face denied access to {repo_path}."
                    )

                if response.status_code >= 500:
                    time.sleep(min(120, 10 * attempt))
                    continue

                response.raise_for_status()

                with open(partial, "wb") as handle:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            handle.write(chunk)

                if partial.stat().st_size <= 0:
                    raise IOError(f"Zero-byte download: {repo_path}")

                partial.replace(destination)
                return destination

        except FileNotFoundError:
            partial.unlink(missing_ok=True)
            raise
        except (requests.RequestException, OSError) as exc:
            partial.unlink(missing_ok=True)
            if attempt == max_attempts:
                raise
            wait = min(120, 10 * attempt)
            print(
                f"Network error for {repo_path}: {exc}\n"
                f"Waiting {wait}s before retry..."
            )
            time.sleep(wait + py_random.uniform(0, 3))

    raise RuntimeError(f"Could not fetch {repo_path}")

print("Repository:", REPO_ID)
print("Revision:", REPO_REVISION)
print("Cache:", CACHE_ROOT)
print("Output root:", DRIVE_PROJECT_ROOT)


Mounted at /content/drive


Repository: hunglc007/ThyroidXL
Revision: b15fe293bd74f1a8a4f05bf88bcdf06a1934125f
Cache: /content/thyroidxl_train_cache
Output root: /content/drive/MyDrive/ThyroidXL_Publication_OneFold


In [5]:
import gc
import json
import random
import re
import hashlib

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
    roc_curve,
)
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
N_FOLDS = 5
FOLD_INDEX = 1

IMAGE_SIZE = 512


EPOCHS_HEAD = 3
MAX_FULL_EPOCHS = 25

BATCH_HEAD = 8
BATCH_FULL = 4

LR_HEAD_STAGE = 5e-4
LR_ENCODER_FULL = 2.5e-5
LR_CLASSIFIER_FULL = 1e-4
LR_DECODER_FULL = 1e-4
WEIGHT_DECAY = 2e-4

MODEL_NAME = "convnext_tiny"
DROP_RATE = 0.20

SEGMENTATION_LOSS_WEIGHT = 0.50
SEGMENTATION_BCE_WEIGHT = 0.50






PRIMARY_PATIENT_THRESHOLD = 0.50
IMAGE_THRESHOLD = 0.50

NUM_WORKERS = 2
PRECACHE_WORKERS = 4

EXPECTED_OFFICIAL_TRAIN_IMAGES = 9541
EXPECTED_OFFICIAL_TRAIN_PATIENTS = 3354
EXPECTED_BENIGN_PATIENTS = 2477
EXPECTED_MALIGNANT_PATIENTS = 877

EXPECTED_FOLD1_TRAIN_IMAGES = 7684
EXPECTED_FOLD1_TRAIN_PATIENTS = 2683
EXPECTED_FOLD1_VAL_IMAGES = 1857
EXPECTED_FOLD1_VAL_PATIENTS = 671

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not enabled. In Colab choose Runtime -> Change runtime type -> GPU."
    )

DEVICE = torch.device("cuda")
USE_AMP = True
CHECK_MODEL_FINITE_EACH_EPOCH = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVELOPMENT_RUN_NAME = (
    f"convnexttiny_thyroidxl_multiscale_patientfold{FOLD_INDEX}_"
    f"512_seed{SEED}_development"
)

FINAL_RUN_NAME = (
    f"convnexttiny_thyroidxl_multiscale_officialtrain9541_"
    f"onefoldselected_512_seed{SEED}_final"
)

print("Development run:", DEVELOPMENT_RUN_NAME)
print("Final run:", FINAL_RUN_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

Development run: convnexttiny_thyroidxl_multiscale_patientfold1_512_seed42_development
Final run: convnexttiny_thyroidxl_multiscale_officialtrain9541_onefoldselected_512_seed42_final
GPU: Tesla T4
PyTorch: 2.11.0+cu128
timm: 1.0.28


In [6]:
annotations_path = cached_remote_file("train/train_annotations.json")

with open(annotations_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

if "images" not in coco or "annotations" not in coco:
    raise RuntimeError("Unexpected ThyroidXL train annotation structure.")

category_map = {
    item["id"]: str(item.get("name", "")).strip()
    for item in coco.get("categories", [])
    if isinstance(item, dict) and "id" in item
}

def patient_id_from_filename(filename):
    stem = Path(str(filename)).stem
    match = re.match(r"^(\d+)(?:_|$)", stem)
    if match is None:
        raise ValueError(f"Cannot derive patient ID from {filename}")
    return str(int(match.group(1)))

def category_to_binary(category_id):
    name = str(category_map.get(category_id, "")).strip().lower()
    if "benign" in name:
        return 0
    if "malignant" in name:
        return 1
    if category_id in (0, 1):
        return int(category_id)
    if str(category_id).strip() in {"0", "1"}:
        return int(category_id)
    return None

image_rows = []
image_id_to_filename = {}

for item in coco["images"]:
    image_id = item["id"]
    filename = Path(str(item["file_name"])).name
    if image_id in image_id_to_filename:
        raise RuntimeError(f"Duplicate image ID: {image_id}")
    image_id_to_filename[image_id] = filename
    image_rows.append({
        "image_id": image_id,
        "filename": filename,
        "patient_id": patient_id_from_filename(filename),
    })

categories_by_image = {}
for ann in coco["annotations"]:
    if not isinstance(ann, dict):
        continue
    iid = ann.get("image_id")
    cid = ann.get("category_id")
    if iid in image_id_to_filename and cid is not None:
        categories_by_image.setdefault(iid, set()).add(cid)

labels = {}
problems = []

for iid, filename in image_id_to_filename.items():
    values = {
        category_to_binary(cid)
        for cid in categories_by_image.get(iid, set())
    }
    values.discard(None)

    if len(values) != 1:
        problems.append(
            (filename, categories_by_image.get(iid, set()), values)
        )
    else:
        labels[iid] = next(iter(values))

if problems:
    raise RuntimeError(
        f"Label derivation failed. Examples: {problems[:10]}"
    )

official_train_df = pd.DataFrame(image_rows)
official_train_df["label"] = (
    official_train_df["image_id"]
    .map(labels)
    .astype(int)
)

if len(official_train_df) != EXPECTED_OFFICIAL_TRAIN_IMAGES:
    raise RuntimeError("Unexpected official training image count.")

if official_train_df["patient_id"].nunique() != EXPECTED_OFFICIAL_TRAIN_PATIENTS:
    raise RuntimeError("Unexpected official training patient count.")

if official_train_df.groupby("patient_id")["label"].nunique().max() != 1:
    raise RuntimeError("Patient-level label inconsistency detected.")

patient_df = (
    official_train_df[["patient_id", "label"]]
    .drop_duplicates()
    .sort_values(
        "patient_id",
        key=lambda x: x.astype(int),
    )
    .reset_index(drop=True)
)

patient_counts = (
    patient_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
)

if patient_counts != {
    0: EXPECTED_BENIGN_PATIENTS,
    1: EXPECTED_MALIGNANT_PATIENTS,
}:
    raise RuntimeError(
        f"Unexpected patient class counts: {patient_counts}"
    )

print("=" * 80)
print("=" * 80)
print("Images:", len(official_train_df))
print("Patients:", official_train_df["patient_id"].nunique())
print("Patient counts:", patient_counts)


Images: 9541
Patients: 3354
Patient counts: {0: 2477, 1: 877}


In [7]:
patient_df = patient_df.copy()
patient_df["fold"] = -1

splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

for fold_zero, (_, val_index) in enumerate(
    splitter.split(
        patient_df["patient_id"],
        patient_df["label"],
    )
):
    patient_df.loc[val_index, "fold"] = fold_zero + 1

patient_to_fold = (
    patient_df
    .set_index("patient_id")["fold"]
    .to_dict()
)

official_train_df["fold"] = (
    official_train_df["patient_id"]
    .map(patient_to_fold)
    .astype(int)
)

development_train_df = (
    official_train_df[
        official_train_df["fold"] != FOLD_INDEX
    ]
    .copy()
    .reset_index(drop=True)
)

development_val_df = (
    official_train_df[
        official_train_df["fold"] == FOLD_INDEX
    ]
    .copy()
    .reset_index(drop=True)
)

overlap = (
    set(development_train_df["patient_id"])
    & set(development_val_df["patient_id"])
)
if overlap:
    raise RuntimeError(
        f"Patient overlap detected: {sorted(overlap)[:10]}"
    )

if len(development_train_df) != EXPECTED_FOLD1_TRAIN_IMAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_FOLD1_TRAIN_IMAGES} Fold-1 train images, "
        f"got {len(development_train_df)}."
    )

if development_train_df["patient_id"].nunique() != EXPECTED_FOLD1_TRAIN_PATIENTS:
    raise RuntimeError("Unexpected Fold-1 train patient count.")

if len(development_val_df) != EXPECTED_FOLD1_VAL_IMAGES:
    raise RuntimeError("Unexpected Fold-1 validation image count.")

if development_val_df["patient_id"].nunique() != EXPECTED_FOLD1_VAL_PATIENTS:
    raise RuntimeError("Unexpected Fold-1 validation patient count.")

print("=" * 80)
print("PATIENT-DISJOINT FOLD 1 VERIFIED")
print("=" * 80)
print(
    "Development train:",
    len(development_train_df),
    "images /",
    development_train_df["patient_id"].nunique(),
    "patients",
)
print(
    "Development validation:",
    len(development_val_df),
    "images /",
    development_val_df["patient_id"].nunique(),
    "patients",
)
print("Patient overlap:", len(overlap))

PATIENT-DISJOINT FOLD 1 VERIFIED
Development train: 7684 images / 2683 patients
Development validation: 1857 images / 671 patients
Patient overlap: 0


In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

required_filenames = sorted(
    official_train_df["filename"].unique()
)
assert len(required_filenames) == EXPECTED_OFFICIAL_TRAIN_IMAGES

def local_pair_paths(filename):
    return (
        CACHE_ROOT / "train" / "images" / filename,
        CACHE_ROOT / "train" / "masks" / filename,
    )

def pair_is_cached(filename):
    image_path, mask_path = local_pair_paths(filename)
    return (
        image_path.is_file()
        and image_path.stat().st_size > 0
        and mask_path.is_file()
        and mask_path.stat().st_size > 0
    )

def fetch_pair(filename):
    cached_remote_file(f"train/images/{filename}")
    cached_remote_file(f"train/masks/{filename}")
    return filename

missing = [
    f
    for f in required_filenames
    if not pair_is_cached(f)
]

print("Required image/mask pairs:", len(required_filenames))
print("Already cached:", len(required_filenames) - len(missing))
print("Remaining:", len(missing))

if missing:
    failures = []

    with ThreadPoolExecutor(
        max_workers=PRECACHE_WORKERS
    ) as executor:
        futures = {
            executor.submit(fetch_pair, filename): filename
            for filename in missing
        }

        with tqdm(
            total=len(futures),
            desc="Caching official-train image/mask pairs",
            unit="pair",
        ) as pbar:
            for future in as_completed(futures):
                filename = futures[future]
                try:
                    future.result()
                except Exception as exc:
                    failures.append(
                        (filename, repr(exc))
                    )
                finally:
                    pbar.update(1)

    if failures:
        raise RuntimeError(
            f"{len(failures)} download failures. "
            f"First examples: {failures[:10]}"
        )

print("Verifying cached pairs...")
problems = []

for filename in tqdm(
    required_filenames,
    desc="Verifying local pairs",
    unit="pair",
):
    image_path, mask_path = local_pair_paths(filename)

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )
    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        problems.append((filename, "unreadable image"))
    elif mask is None:
        problems.append((filename, "unreadable mask"))
    elif image.shape[:2] != mask.shape[:2]:
        problems.append((filename, "shape mismatch"))
    elif not (mask > 0).any():
        problems.append((filename, "empty mask"))

if problems:
    raise RuntimeError(
        f"Pair verification failed. First examples: {problems[:10]}"
    )

print(
    f"✅ Verified {len(required_filenames):,} official-training pairs."
)


Required image/mask pairs: 9541
Already cached: 0
Remaining: 9541


Caching official-train image/mask pairs:   0%|          | 0/9541 [00:00<?, ?pair/s]

Verifying cached pairs...


Verifying local pairs:   0%|          | 0/9541 [00:00<?, ?pair/s]

✅ Verified 9,541 official-training pairs.


In [9]:
train_transform = A.Compose([
    A.LongestMaxSize(
        max_size=IMAGE_SIZE,
        area_for_downscale="image",
    ),
    A.PadIfNeeded(
        min_height=IMAGE_SIZE,
        min_width=IMAGE_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=0,
    ),
    A.HorizontalFlip(p=0.5),
    A.Affine(
        scale=(0.92, 1.06),
        translate_percent=(-0.03, 0.03),
        rotate=(-10, 10),
        shear=(-2, 2),
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=0,
        p=0.70,
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.12,
        contrast_limit=0.12,
        p=0.40,
    ),
    A.RandomGamma(
        gamma_limit=(85, 115),
        p=0.20,
    ),
    A.GaussianBlur(
        blur_limit=(3, 5),
        sigma_limit=(0.1, 1.0),
        p=0.12,
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    ),
    ToTensorV2(),
], seed=SEED, strict=True)

val_transform = A.Compose([
    A.LongestMaxSize(
        max_size=IMAGE_SIZE,
        area_for_downscale="image",
    ),
    A.PadIfNeeded(
        min_height=IMAGE_SIZE,
        min_width=IMAGE_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=0,
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    ),
    ToTensorV2(),
], seed=SEED, strict=True)

In [10]:
class ThyroidXLClassificationSegmentationDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        filename = row['filename']


        image_path = CACHE_ROOT / 'train' / 'images' / filename
        mask_path = CACHE_ROOT / 'train' / 'masks' / filename

        if not image_path.is_file() or image_path.stat().st_size == 0:
            raise FileNotFoundError(
                f'Missing cached training image: {image_path}. '
                'Re-run Section 6.1 pre-cache before training.'
            )
        if not mask_path.is_file() or mask_path.stat().st_size == 0:
            raise FileNotFoundError(
                f'Missing cached training mask: {mask_path}. '
                'Re-run Section 6.1 pre-cache before training.'
            )

        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f'OpenCV could not read cached image: {image_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f'OpenCV could not read cached mask: {mask_path}')

        if mask.shape[:2] != image.shape[:2]:
            raise ValueError(
                f'Image/mask shape mismatch for {filename}: '
                f'{image.shape[:2]} vs {mask.shape[:2]}'
            )

        mask = (mask > 0).astype(np.uint8)
        if not mask.any():
            raise ValueError(f'Empty nodule mask for {filename}')

        transformed = self.transform(image=image, mask=mask)

        image_tensor = transformed['image'].float()
        mask_tensor = transformed['mask']
        if mask_tensor.ndim == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
        mask_tensor = (mask_tensor > 0).float()

        return {
            'image': image_tensor,
            'mask': mask_tensor,
            'label': torch.tensor(float(row['label']), dtype=torch.float32),
            'filename': filename,
            'patient_id': str(row['patient_id']),
        }


class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SkipFusionBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.refine = ConvBNAct(
            in_channels + skip_channels,
            out_channels,
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        x = torch.cat([x, skip], dim=1)
        return self.refine(x)


class UpsampleRefineBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.refine = ConvBNAct(in_channels, out_channels)

    def forward(self, x, target_size):
        x = F.interpolate(
            x,
            size=target_size,
            mode="bilinear",
            align_corners=False,
        )
        return self.refine(x)


class ConvNeXtTinyMultiscaleDiceBCE(nn.Module):

    def __init__(self, image_size=IMAGE_SIZE):
        super().__init__()
        self.image_size = int(image_size)

        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=True,
            num_classes=1,
            drop_rate=DROP_RATE,
        )


        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            dummy = torch.zeros(
                1,
                3,
                self.image_size,
                self.image_size,
            )
            final_feature, skip_features = self._encode(dummy)
        if was_training:
            self.backbone.train()

        final_channels = int(final_feature.shape[1])
        skip32_channels = int(skip_features[32].shape[1])
        skip64_channels = int(skip_features[64].shape[1])
        skip128_channels = int(skip_features[128].shape[1])

        expected_final = (
            self.image_size // 32,
            self.image_size // 32,
        )
        if tuple(final_feature.shape[-2:]) != expected_final:
            raise RuntimeError(
                "Unexpected ConvNeXt-Tiny final feature resolution: "
                f"{tuple(final_feature.shape)}"
            )



        self.decoder_bottleneck = ConvBNAct(
            final_channels,
            192,
        )
        self.decoder_skip32 = SkipFusionBlock(
            192,
            skip32_channels,
            128,
        )
        self.decoder_skip64 = SkipFusionBlock(
            128,
            skip64_channels,
            96,
        )
        self.decoder_skip128 = SkipFusionBlock(
            96,
            skip128_channels,
            64,
        )
        self.decoder_up256 = UpsampleRefineBlock(
            64,
            32,
        )
        self.decoder_up512 = UpsampleRefineBlock(
            32,
            16,
        )
        self.segmentation_output = nn.Conv2d(
            16,
            1,
            kernel_size=1,
        )

        self.inferred_feature_channels = {
            "final": final_channels,
            "skip32": skip32_channels,
            "skip64": skip64_channels,
            "skip128": skip128_channels,
        }

    def segmentation_decoder_modules(self):
        return [
            self.decoder_bottleneck,
            self.decoder_skip32,
            self.decoder_skip64,
            self.decoder_skip128,
            self.decoder_up256,
            self.decoder_up512,
            self.segmentation_output,
        ]

    def segmentation_decoder_parameters(self):
        for module in self.segmentation_decoder_modules():
            yield from module.parameters()

    def _encode(self, image):

        x = self.backbone.stem(image)

        stage_outputs = []
        for stage in self.backbone.stages:
            x = stage(x)
            stage_outputs.append(x)

        skips = {}
        for target in (32, 64, 128):
            candidates = [
                feature
                for feature in stage_outputs
                if tuple(feature.shape[-2:]) == (target, target)
            ]
            if not candidates:
                available = sorted({
                    tuple(feature.shape[-2:])
                    for feature in stage_outputs
                })
                raise RuntimeError(
                    f"No ConvNeXt-Tiny skip feature found at {target}x{target}. "
                    f"Available stage resolutions: {available}"
                )
            skips[target] = candidates[-1]


        x = self.backbone.norm_pre(x)
        return x, skips

    def forward(self, image):
        final_feature, skips = self._encode(image)

        classification_logits = self.backbone.forward_head(
            final_feature
        ).flatten()

        x = self.decoder_bottleneck(final_feature)
        x = self.decoder_skip32(x, skips[32])
        x = self.decoder_skip64(x, skips[64])
        x = self.decoder_skip128(x, skips[128])
        x = self.decoder_up256(
            x,
            (self.image_size // 2, self.image_size // 2),
        )
        x = self.decoder_up512(
            x,
            (self.image_size, self.image_size),
        )

        segmentation_logits = self.segmentation_output(x)

        if segmentation_logits.shape[-2:] != image.shape[-2:]:
            segmentation_logits = F.interpolate(
                segmentation_logits,
                size=image.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        return classification_logits, segmentation_logits


def make_model():
    model = ConvNeXtTinyMultiscaleDiceBCE(
        image_size=IMAGE_SIZE
    ).to(DEVICE)
    return model

def make_loaders(train_df, val_df=None):
    train_df = train_df.sort_values(['patient_id', 'filename']).reset_index(drop=True)
    train_dataset = ThyroidXLClassificationSegmentationDataset(
        train_df,
        train_transform,
    )

    train_loader_head = DataLoader(
        train_dataset,
        batch_size=BATCH_HEAD,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        generator=torch.Generator().manual_seed(SEED),
        drop_last=False,
    )

    train_loader_full = DataLoader(
        train_dataset,
        batch_size=BATCH_FULL,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        generator=torch.Generator().manual_seed(SEED),
        drop_last=False,
    )

    val_loader = None

    if val_df is not None:
        val_df = val_df.sort_values(['patient_id', 'filename']).reset_index(drop=True)
        val_dataset = ThyroidXLClassificationSegmentationDataset(
            val_df,
            val_transform,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_FULL,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=(NUM_WORKERS > 0),
            drop_last=False,
        )

    return train_loader_head, train_loader_full, val_loader

In [11]:
classification_criterion = nn.BCEWithLogitsLoss()
segmentation_bce_criterion = nn.BCEWithLogitsLoss()


def soft_dice_loss(logits, targets, eps=1e-6):
    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)

    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)

    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


def hard_dice_per_sample(logits, targets, threshold=0.5, eps=1e-6):
    predictions = (torch.sigmoid(logits) >= threshold).float()
    dims = (1, 2, 3)

    intersection = (predictions * targets).sum(dim=dims)
    denominator = predictions.sum(dim=dims) + targets.sum(dim=dims)

    return (2.0 * intersection + eps) / (denominator + eps)


def hard_iou_per_sample(logits, targets, threshold=0.5, eps=1e-6):
    predictions = (torch.sigmoid(logits) >= threshold).float()
    dims = (1, 2, 3)

    intersection = (predictions * targets).sum(dim=dims)
    union = predictions.sum(dim=dims) + targets.sum(dim=dims) - intersection
    return (intersection + eps) / (union + eps)


def classification_metrics(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) else float('nan')

    return {
        'auc': float(roc_auc_score(labels, probabilities)),
        'auprc': float(average_precision_score(labels, probabilities)),
        'accuracy': float(accuracy_score(labels, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'precision': float(precision_score(labels, predictions, zero_division=0)),
        'f1': float(f1_score(labels, predictions, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def aggregate_patient_predictions(patient_ids, labels, probabilities):
    frame = pd.DataFrame({
        'patient_id': [str(x) for x in patient_ids],
        'label': np.asarray(labels, dtype=np.int64),
        'probability_malignant': np.asarray(probabilities, dtype=np.float64),
    })

    label_consistency = frame.groupby('patient_id')['label'].nunique()
    if label_consistency.max() != 1:
        raise RuntimeError('A patient has inconsistent classification labels.')

    patient_frame = (
        frame.groupby('patient_id', as_index=False)
        .agg(
            label=('label', 'first'),
            probability_malignant=('probability_malignant', 'mean'),
            n_frames=('probability_malignant', 'size'),
        )
    )
    return patient_frame


def aggregate_patient_weighted_majority_vote(
    patient_ids,
    labels,
    probabilities,
    image_threshold=0.5,
):
    frame = pd.DataFrame({
        "patient_id": [str(x) for x in patient_ids],
        "label": np.asarray(labels, dtype=np.int64),
        "probability_malignant": np.asarray(
            probabilities,
            dtype=np.float64,
        ),
    })

    if frame.groupby("patient_id")["label"].nunique().max() != 1:
        raise RuntimeError(
            "A patient has inconsistent classification labels."
        )

    frame["image_prediction"] = (
        frame["probability_malignant"] >= image_threshold
    ).astype(int)

    frame["benign_vote_weight"] = np.where(
        frame["image_prediction"] == 0,
        1.0 - frame["probability_malignant"],
        0.0,
    )
    frame["malignant_vote_weight"] = np.where(
        frame["image_prediction"] == 1,
        frame["probability_malignant"],
        0.0,
    )

    patient = (
        frame.groupby("patient_id", as_index=False)
        .agg(
            label=("label", "first"),
            benign_vote_weight=("benign_vote_weight", "sum"),
            malignant_vote_weight=("malignant_vote_weight", "sum"),
            n_frames=("probability_malignant", "size"),
        )
    )

    patient["prediction_wmv"] = (
        patient["malignant_vote_weight"]
        > patient["benign_vote_weight"]
    ).astype(int)


    ties = (
        patient["malignant_vote_weight"]
        == patient["benign_vote_weight"]
    )
    if ties.any():
        mean_scores = aggregate_patient_predictions(
            patient_ids,
            labels,
            probabilities,
        ).set_index("patient_id")["probability_malignant"]
        patient.loc[ties, "prediction_wmv"] = (
            patient.loc[ties, "patient_id"]
            .map(mean_scores)
            .ge(0.5)
            .astype(int)
        )

    return patient



def assert_finite_tensor(name, tensor):
    if not torch.isfinite(tensor).all():
        finite_fraction = float(torch.isfinite(tensor).float().mean().item())
        raise FloatingPointError(
            f"Non-finite values detected in {name}; "
            f"finite_fraction={finite_fraction:.6f}."
        )


def assert_model_parameters_finite(model, context="model"):
    bad = []
    with torch.no_grad():
        for name, tensor in model.state_dict().items():
            if torch.is_floating_point(tensor) and not torch.isfinite(tensor).all():
                bad.append(name)
                if len(bad) >= 10:
                    break
    if bad:
        raise FloatingPointError(
            f"Non-finite model parameters/buffers detected after {context}: {bad}"
        )


def copy_state_dict(model):
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def run_epoch(
    model,
    dataloader,
    train,
    optimizer=None,
    scheduler=None,
    scaler=None,
    frozen_head_stage=False,
):
    model.train(train)

    if train and frozen_head_stage:
        for module in model.backbone.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()

    total_loss_sum = 0.0
    classification_loss_sum = 0.0
    segmentation_loss_sum = 0.0
    segmentation_dice_loss_sum = 0.0
    segmentation_bce_loss_sum = 0.0

    dice_sum = 0.0
    iou_sum = 0.0
    predicted_fraction_sum = 0.0
    expert_fraction_sum = 0.0

    labels_all = []
    probabilities_all = []
    patient_ids_all = []

    for batch in tqdm(dataloader, leave=False):
        images = batch['image'].to(DEVICE, non_blocking=True)
        masks = batch['mask'].to(DEVICE, non_blocking=True)
        labels = batch['label'].to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                classification_logits, segmentation_logits = model(images)

                classification_loss = classification_criterion(
                    classification_logits,
                    labels,
                )

                segmentation_dice_loss = soft_dice_loss(
                    segmentation_logits,
                    masks,
                )
                segmentation_bce_loss = segmentation_bce_criterion(
                    segmentation_logits,
                    masks,
                )

                segmentation_loss = (
                    segmentation_dice_loss
                    + SEGMENTATION_BCE_WEIGHT * segmentation_bce_loss
                )

                total_loss = (
                    classification_loss
                    + SEGMENTATION_LOSS_WEIGHT * segmentation_loss
                )



                assert_finite_tensor(
                    "classification_logits",
                    classification_logits,
                )
                assert_finite_tensor(
                    "segmentation_logits",
                    segmentation_logits,
                )
                assert_finite_tensor(
                    "classification_loss",
                    classification_loss,
                )
                assert_finite_tensor(
                    "segmentation_loss",
                    segmentation_loss,
                )
                assert_finite_tensor(
                    "total_loss",
                    total_loss,
                )

            if train:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()

                if scheduler is not None and scaler.get_scale() >= previous_scale:
                    scheduler.step()

        probabilities = torch.sigmoid(classification_logits)
        assert_finite_tensor(
            "classification_probabilities",
            probabilities,
        )

        with torch.no_grad():
            segmentation_probabilities = torch.sigmoid(segmentation_logits)
            segmentation_predictions = (segmentation_probabilities >= 0.5).float()

            batch_dice = hard_dice_per_sample(segmentation_logits, masks)
            batch_iou = hard_iou_per_sample(segmentation_logits, masks)

            predicted_fraction_per_case = segmentation_predictions.mean(dim=(1, 2, 3))
            expert_fraction_per_case = masks.mean(dim=(1, 2, 3))

        batch_size = labels.shape[0]

        total_loss_sum += float(total_loss.item()) * batch_size
        classification_loss_sum += float(classification_loss.item()) * batch_size
        segmentation_loss_sum += float(segmentation_loss.item()) * batch_size
        segmentation_dice_loss_sum += float(segmentation_dice_loss.item()) * batch_size
        segmentation_bce_loss_sum += float(segmentation_bce_loss.item()) * batch_size

        dice_sum += float(batch_dice.sum().item())
        iou_sum += float(batch_iou.sum().item())
        predicted_fraction_sum += float(predicted_fraction_per_case.sum().item())
        expert_fraction_sum += float(expert_fraction_per_case.sum().item())

        labels_all.extend(labels.detach().cpu().numpy().tolist())
        probabilities_all.extend(probabilities.detach().cpu().numpy().tolist())
        patient_ids_all.extend([str(x) for x in batch['patient_id']])

    labels_all = np.asarray(labels_all)
    probabilities_all = np.asarray(probabilities_all)

    image_metrics = classification_metrics(
        labels_all,
        probabilities_all,
        threshold=0.5,
    )

    patient_predictions = aggregate_patient_predictions(
        patient_ids_all,
        labels_all,
        probabilities_all,
    )
    patient_metrics = classification_metrics(
        patient_predictions['label'].to_numpy(),
        patient_predictions['probability_malignant'].to_numpy(),
        threshold=0.5,
    )


    metrics = dict(patient_metrics)
    metrics.update({f'patient_{k}': v for k, v in patient_metrics.items()})
    metrics.update({f'image_{k}': v for k, v in image_metrics.items()})

    count = len(dataloader.dataset)
    metrics['n_patients'] = int(len(patient_predictions))
    metrics['loss'] = total_loss_sum / count
    metrics['classification_loss'] = classification_loss_sum / count
    metrics['segmentation_loss'] = segmentation_loss_sum / count
    metrics['segmentation_dice_loss'] = segmentation_dice_loss_sum / count
    metrics['segmentation_bce_loss'] = segmentation_bce_loss_sum / count
    metrics['segmentation_dice'] = dice_sum / count
    metrics['segmentation_iou'] = iou_sum / count
    metrics['predicted_mask_fraction'] = predicted_fraction_sum / count
    metrics['expert_mask_fraction'] = expert_fraction_sum / count

    return metrics, labels_all, probabilities_all


def print_epoch(prefix, metrics):
    print(
        f"{prefix} "
        f"loss={metrics['loss']:.4f} | "
        f"cls={metrics['classification_loss']:.4f} | "
        f"seg={metrics['segmentation_loss']:.4f} | "
        f"Dice={metrics['segmentation_dice']:.4f} | "
        f"IoU={metrics['segmentation_iou']:.4f} | "
        f"PredMask={metrics['predicted_mask_fraction']:.3f} | "
        f"GTMask={metrics['expert_mask_fraction']:.3f} | "
        f"PatientAUC={metrics['patient_auc']:.4f} | "
        f"PatientAUPRC={metrics['patient_auprc']:.4f} | "
        f"ImageAUC={metrics['image_auc']:.4f} | "
        f"Patients={metrics['n_patients']}"
    )

In [12]:
development_model = make_model()

dev_train_head, dev_train_full, dev_val_loader = make_loaders(
    development_train_df,
    development_val_df,
)



development_val_order = (
    development_val_df
    .sort_values(["patient_id", "filename"])
    .reset_index(drop=True)
)
development_val_patient_ids = (
    development_val_order["patient_id"]
    .astype(str)
    .to_numpy()
)

for parameter in development_model.backbone.parameters():
    parameter.requires_grad = False

classifier_module = development_model.backbone.get_classifier()
for parameter in classifier_module.parameters():
    parameter.requires_grad = True

for parameter in development_model.segmentation_decoder_parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW(
    [p for p in development_model.parameters() if p.requires_grad],
    lr=LR_HEAD_STAGE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR_HEAD_STAGE,
    steps_per_epoch=len(dev_train_head),
    epochs=EPOCHS_HEAD,
    pct_start=0.20,
    div_factor=25.0,
    final_div_factor=1e4,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

development_history = []

best_patient_auc = -np.inf
best_state = None
best_phase = None
best_epoch = None
best_val_metrics = None
best_labels = None
best_probabilities = None
best_patient_ids = None

best_head_auc = -np.inf
best_head_epoch = None
best_head_state = None

for epoch in range(1, EPOCHS_HEAD + 1):
    train_metrics, _, _ = run_epoch(
        development_model,
        dev_train_head,
        train=True,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        frozen_head_stage=True,
    )

    val_metrics, val_labels, val_probabilities = run_epoch(
        development_model,
        dev_val_loader,
        train=False,
    )

    if CHECK_MODEL_FINITE_EACH_EPOCH:
        assert_model_parameters_finite(
            development_model,
            context=f"development head epoch {epoch}",
        )

    if len(val_probabilities) != len(development_val_order):
        raise RuntimeError(
            "Validation prediction count does not match Fold-1 validation frame."
        )

    patient_predictions = aggregate_patient_predictions(
        development_val_patient_ids,
        val_labels,
        val_probabilities,
    )
    patient_auc = float(
        roc_auc_score(
            patient_predictions["label"],
            patient_predictions["probability_malignant"],
        )
    )

    print(
        f"Head epoch {epoch:02d}/{EPOCHS_HEAD} | "
        f"patient AUC={patient_auc:.6f}"
    )
    print_epoch("  Train:", train_metrics)
    print_epoch("  Val:  ", val_metrics)

    development_history.append({
        "phase": "head",
        "epoch": epoch,
        "val_patient_auc": patient_auc,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    })

    if patient_auc > best_head_auc:
        best_head_auc = patient_auc
        best_head_epoch = int(epoch)
        best_head_state = copy_state_dict(development_model)


    if patient_auc > best_patient_auc:
        best_patient_auc = patient_auc
        best_state = copy_state_dict(development_model)
        best_phase = "head"
        best_epoch = int(epoch)
        best_val_metrics = dict(val_metrics)
        best_labels = np.asarray(val_labels).copy()
        best_probabilities = np.asarray(val_probabilities).copy()
        best_patient_ids = development_val_patient_ids.copy()

if best_head_state is None:
    raise RuntimeError("No head-stage checkpoint was selected.")

print()
print(
    "Best head-stage validation patient AUC:",
    f"{best_head_auc:.6f}",
)
print("Best head-stage epoch:", best_head_epoch)

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/961 [00:00<?, ?it/s]

  0%|          | 0/465 [00:00<?, ?it/s]

Head epoch 01/3 | patient AUC=0.719758
  Train: loss=1.0062 | cls=0.5817 | seg=0.8490 | Dice=0.6471 | IoU=0.5211 | PredMask=0.103 | GTMask=0.059 | PatientAUC=0.5941 | PatientAUPRC=0.3139 | ImageAUC=0.5719 | Patients=2683
  Val:   loss=0.7298 | cls=0.5426 | seg=0.3742 | Dice=0.7967 | IoU=0.6877 | PredMask=0.050 | GTMask=0.054 | PatientAUC=0.7198 | PatientAUPRC=0.4473 | ImageAUC=0.6806 | Patients=671


...

  0%|          | 0/465 [00:00<?, ?it/s]

Head epoch 02/3 | patient AUC=0.743168
  Train: loss=0.6662 | cls=0.5357 | seg=0.2609 | Dice=0.8209 | IoU=0.7173 | PredMask=0.058 | GTMask=0.059 | PatientAUC=0.7161 | PatientAUPRC=0.4306 | ImageAUC=0.6763 | Patients=2683
  Val:   loss=0.6528 | cls=0.5434 | seg=0.2189 | Dice=0.8304 | IoU=0.7300 | PredMask=0.056 | GTMask=0.054 | PatientAUC=0.7432 | PatientAUPRC=0.4811 | ImageAUC=0.7069 | Patients=671


  0%|          | 0/961 [00:00<?, ?it/s]

  0%|          | 0/465 [00:00<?, ?it/s]

Head epoch 03/3 | patient AUC=0.744171
  Train: loss=0.6176 | cls=0.5242 | seg=0.1868 | Dice=0.8582 | IoU=0.7670 | PredMask=0.057 | GTMask=0.059 | PatientAUC=0.7396 | PatientAUPRC=0.4526 | ImageAUC=0.7006 | Patients=2683
  Val:   loss=0.6288 | cls=0.5299 | seg=0.1977 | Dice=0.8456 | IoU=0.7529 | PredMask=0.053 | GTMask=0.054 | PatientAUC=0.7442 | PatientAUPRC=0.4836 | ImageAUC=0.7074 | Patients=671

Best head-stage validation patient AUC: 0.744171
Best head-stage epoch: 3


In [13]:
development_model.load_state_dict(
    best_head_state,
    strict=True,
)
development_model.to(DEVICE)

for parameter in development_model.backbone.parameters():
    parameter.requires_grad = True

classifier_module = development_model.backbone.get_classifier()
classifier_ids = {
    id(parameter)
    for parameter in classifier_module.parameters()
}

encoder_parameters = [
    p
    for p in development_model.backbone.parameters()
    if id(p) not in classifier_ids
]
classifier_parameters = list(
    classifier_module.parameters()
)
decoder_parameters = list(
    development_model.segmentation_decoder_parameters()
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_parameters,
            "lr": LR_ENCODER_FULL,
        },
        {
            "params": classifier_parameters,
            "lr": LR_CLASSIFIER_FULL,
        },
        {
            "params": decoder_parameters,
            "lr": LR_DECODER_FULL,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[
        LR_ENCODER_FULL,
        LR_CLASSIFIER_FULL,
        LR_DECODER_FULL,
    ],
    steps_per_epoch=len(dev_train_full),
    epochs=MAX_FULL_EPOCHS,
    pct_start=0.30,
    div_factor=10.0,
    final_div_factor=1000.0,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

best_full_auc = -np.inf
best_full_epoch = None

for epoch in range(1, MAX_FULL_EPOCHS + 1):
    train_metrics, _, _ = run_epoch(
        development_model,
        dev_train_full,
        train=True,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
    )

    val_metrics, val_labels, val_probabilities = run_epoch(
        development_model,
        dev_val_loader,
        train=False,
    )

    if CHECK_MODEL_FINITE_EACH_EPOCH:
        assert_model_parameters_finite(
            development_model,
            context=f"development full epoch {epoch}",
        )

    if len(val_probabilities) != len(development_val_order):
        raise RuntimeError(
            "Validation prediction count/order check failed."
        )

    patient_predictions = aggregate_patient_predictions(
        development_val_patient_ids,
        val_labels,
        val_probabilities,
    )

    patient_auc = float(
        roc_auc_score(
            patient_predictions["label"],
            patient_predictions["probability_malignant"],
        )
    )

    print(
        f"Full epoch {epoch:02d}/{MAX_FULL_EPOCHS} | "
        f"patient AUC={patient_auc:.6f}"
    )
    print_epoch("  Train:", train_metrics)
    print_epoch("  Val:  ", val_metrics)

    development_history.append({
        "phase": "full",
        "epoch": epoch,
        "val_patient_auc": patient_auc,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    })

    if patient_auc > best_full_auc:
        best_full_auc = patient_auc
        best_full_epoch = int(epoch)


    if patient_auc > best_patient_auc:
        best_patient_auc = patient_auc
        best_state = copy_state_dict(development_model)
        best_phase = "full"
        best_epoch = int(epoch)
        best_val_metrics = dict(val_metrics)
        best_labels = np.asarray(val_labels).copy()
        best_probabilities = np.asarray(
            val_probabilities
        ).copy()
        best_patient_ids = (
            development_val_patient_ids.copy()
        )

if best_state is None:
    raise RuntimeError(
        "No development checkpoint was selected."
    )

selected_head_epochs = int(best_head_epoch)
selected_full_epochs = (
    int(best_epoch)
    if best_phase == "full"
    else 0
)

print()
print("=" * 80)
print("=" * 80)
print(
    "Selected global checkpoint:",
    best_phase,
    "epoch",
    best_epoch,
)
print(
    "Best validation patient AUC:",
    f"{best_patient_auc:.6f}",
)
print(
    "Final refit head epochs:",
    selected_head_epochs,
)
print(
    "Final refit full-stage epochs:",
    selected_full_epochs,
)


  0%|          | 0/1921 [00:00<?, ?it/s]

  0%|          | 0/465 [00:00<?, ?it/s]

Full epoch 01/25 | patient AUC=0.923099
  Train: loss=0.5841 | cls=0.4921 | seg=0.1840 | Dice=0.8597 | IoU=0.7693 | PredMask=0.057 | GTMask=0.059 | PatientAUC=0.8530 | PatientAUPRC=0.6746 | ImageAUC=0.7987 | Patients=2683
  Val:   loss=0.5440 | cls=0.4526 | seg=0.1827 | Dice=0.8573 | IoU=0.7665 | PredMask=0.052 | GTMask=0.054 | PatientAUC=0.9231 | PatientAUPRC=0.8091 | ImageAUC=0.8924 | Patients=671


  0%|          | 0/1921 [00:00<?, ?it/s]

  0%|          | 0/465 [00:00<?, ?it/s]

...

  0%|          | 0/465 [00:00<?, ?it/s]

Full epoch 24/25 | patient AUC=0.947010
  Train: loss=0.0240 | cls=0.0001 | seg=0.0478 | Dice=0.9602 | IoU=0.9244 | PredMask=0.058 | GTMask=0.059 | PatientAUC=1.0000 | PatientAUPRC=1.0000 | ImageAUC=1.0000 | Patients=2683
  Val:   loss=1.6576 | cls=1.5890 | seg=0.1372 | Dice=0.8909 | IoU=0.8143 | PredMask=0.053 | GTMask=0.054 | PatientAUC=0.9470 | PatientAUPRC=0.8517 | ImageAUC=0.9132 | Patients=671


  0%|          | 0/1921 [00:00<?, ?it/s]

  0%|          | 0/465 [00:00<?, ?it/s]

Full epoch 25/25 | patient AUC=0.946740
  Train: loss=0.0244 | cls=0.0008 | seg=0.0473 | Dice=0.9607 | IoU=0.9252 | PredMask=0.058 | GTMask=0.059 | PatientAUC=1.0000 | PatientAUPRC=1.0000 | ImageAUC=1.0000 | Patients=2683
  Val:   loss=1.6580 | cls=1.5896 | seg=0.1369 | Dice=0.8909 | IoU=0.8144 | PredMask=0.052 | GTMask=0.054 | PatientAUC=0.9467 | PatientAUPRC=0.8510 | ImageAUC=0.9113 | Patients=671

Selected global checkpoint: full epoch 11
Best validation patient AUC: 0.960196
Final refit head epochs: 3
Final refit full-stage epochs: 11


In [14]:
best_patient_predictions = aggregate_patient_predictions(
    best_patient_ids,
    best_labels,
    best_probabilities,
)

fpr, tpr, thresholds = roc_curve(
    best_patient_predictions["label"],
    best_patient_predictions["probability_malignant"],
)

finite = np.isfinite(thresholds)
if not finite.any():
    raise RuntimeError(
        "No finite threshold available for patient-level ROC."
    )

youden = tpr[finite] - fpr[finite]
selected_threshold = float(
    thresholds[finite][np.argmax(youden)]
)




validation_image_predictions = (
    development_val_order[
        ["filename", "patient_id", "label", "fold"]
    ]
    .copy()
)
validation_image_predictions[
    "probability_malignant"
] = best_probabilities
validation_image_predictions[
    "prediction_at_0_5"
] = (
    validation_image_predictions[
        "probability_malignant"
    ] >= IMAGE_THRESHOLD
).astype(int)

validation_patient_predictions = (
    best_patient_predictions.copy()
)
validation_patient_predictions["fold"] = FOLD_INDEX
validation_patient_predictions[
    "prediction_at_0_5"
] = (
    validation_patient_predictions[
        "probability_malignant"
    ] >= PRIMARY_PATIENT_THRESHOLD
).astype(int)
validation_patient_predictions[
    "prediction_at_development_selected_threshold"
] = (
    validation_patient_predictions[
        "probability_malignant"
    ] >= selected_threshold
).astype(int)

wmv_predictions = aggregate_patient_weighted_majority_vote(
    best_patient_ids,
    best_labels,
    best_probabilities,
    image_threshold=IMAGE_THRESHOLD,
)
validation_patient_predictions = (
    validation_patient_predictions.merge(
        wmv_predictions[
            [
                "patient_id",
                "benign_vote_weight",
                "malignant_vote_weight",
                "prediction_wmv",
            ]
        ],
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
)

development_checkpoint_path = (
    MODEL_DIR
    / f"{DEVELOPMENT_RUN_NAME}_best.pt"
)

development_checkpoint = {
    "status": (
        "THYROIDXL_CONVNEXTTINY_ONEFOLD_"
        "DEVELOPMENT_SELECTED"
    ),
    "dataset": "ThyroidXL",
    "dataset_repo": REPO_ID,
    "dataset_revision": REPO_REVISION,
    "official_test_accessed": False,
    "split_level": "patient",
    "fold_index": FOLD_INDEX,
    "n_folds": N_FOLDS,
    "patient_split_seed": SEED,
    "training_images": int(len(development_train_df)),
    "training_patients": int(
        development_train_df["patient_id"].nunique()
    ),
    "validation_images": int(len(development_val_df)),
    "validation_patients": int(
        development_val_df["patient_id"].nunique()
    ),
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "amp_enabled": bool(USE_AMP),
    "numerical_safety": {
        "fail_on_nonfinite_logits_losses_probabilities": True,
        "check_model_parameters_each_epoch": bool(
            CHECK_MODEL_FINITE_EACH_EPOCH
        ),
    },
    "best_phase": best_phase,
    "best_epoch": int(best_epoch),
    "best_head_epoch": int(best_head_epoch),
    "selected_head_epochs_for_final_refit": (
        selected_head_epochs
    ),
    "selected_full_epochs_for_final_refit": (
        selected_full_epochs
    ),
    "best_validation_patient_auc": float(
        best_patient_auc
    ),
    "best_validation_image_auc": float(
        best_val_metrics["image_auc"]
    ),
    "best_validation_segmentation_dice": float(
        best_val_metrics["segmentation_dice"]
    ),
    "best_validation_segmentation_iou": float(
        best_val_metrics["segmentation_iou"]
    ),
    "primary_patient_threshold": (
        PRIMARY_PATIENT_THRESHOLD
    ),
    "development_selected_patient_threshold": (
        selected_threshold
    ),

    "validation_patient_threshold": selected_threshold,
    "patient_threshold_method": (
        "Youden J on Fold-1 patient-level validation "
        "mean-probability scores"
    ),
    "patient_aggregation_primary": (
        "mean malignant probability across all frames"
    ),
    "patient_aggregation_benchmark_secondary": (
        "confidence-weighted majority voting as described "
        "in the ThyroidXL benchmark paper"
    ),
    "state_dict": best_state,
}

torch.save(
    development_checkpoint,
    development_checkpoint_path,
)

development_history_path = (
    RESULTS_DIR
    / f"{DEVELOPMENT_RUN_NAME}_history.csv"
)
pd.DataFrame(development_history).to_csv(
    development_history_path,
    index=False,
)

fold_assignments_path = (
    RESULTS_DIR
    / f"{DEVELOPMENT_RUN_NAME}_patient_fold_assignments.csv"
)
patient_df[
    ["patient_id", "label", "fold"]
].to_csv(
    fold_assignments_path,
    index=False,
)

validation_image_predictions_path = (
    RESULTS_DIR
    / f"{DEVELOPMENT_RUN_NAME}_validation_image_predictions.csv"
)
validation_image_predictions.to_csv(
    validation_image_predictions_path,
    index=False,
)

validation_patient_predictions_path = (
    RESULTS_DIR
    / f"{DEVELOPMENT_RUN_NAME}_validation_patient_predictions.csv"
)
validation_patient_predictions.to_csv(
    validation_patient_predictions_path,
    index=False,
)

development_manifest = {
    key: value
    for key, value in development_checkpoint.items()
    if key != "state_dict"
}

development_manifest["versions"] = {
    "torch": torch.__version__,
    "timm": timm.__version__,
    "albumentations": A.__version__,
}

development_manifest_path = (
    RESULTS_DIR
    / f"{DEVELOPMENT_RUN_NAME}_manifest.json"
)
development_manifest_path.write_text(
    json.dumps(
        development_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 80)
print("=" * 80)
print(
    "Best checkpoint:",
    best_phase,
    "epoch",
    best_epoch,
)
print(
    "Best patient AUC:",
    f"{best_patient_auc:.6f}",
)
print(
    "Primary final-model patient threshold:",
    PRIMARY_PATIENT_THRESHOLD,
)
print(
    "Development-selected secondary threshold:",
    f"{selected_threshold:.6f}",
)
print("Development checkpoint:", development_checkpoint_path)
print("Fold assignments:", fold_assignments_path)
print(
    "Validation image predictions:",
    validation_image_predictions_path,
)
print(
    "Validation patient predictions:",
    validation_patient_predictions_path,
)


Best checkpoint: full epoch 11
Best patient AUC: 0.960196
Primary final-model patient threshold: 0.5
Development-selected secondary threshold: 0.249004
Development checkpoint: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/Models/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_patientfold1_512_seed42_development_best.pt
Fold assignments: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/results/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_patientfold1_512_seed42_development_patient_fold_assignments.csv
Validation image predictions: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/results/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_patientfold1_512_seed42_development_validation_image_predictions.csv
Validation patient predictions: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/results/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_patientfold1_512_seed42_development_validation_patient_predictions.csv


In [15]:


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

final_model = make_model()

final_train_head, final_train_full, _ = make_loaders(
    official_train_df,
    None,
)

final_history = []




for parameter in final_model.backbone.parameters():
    parameter.requires_grad = False

classifier_module = final_model.backbone.get_classifier()
for parameter in classifier_module.parameters():
    parameter.requires_grad = True

for parameter in final_model.segmentation_decoder_parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW(
    [p for p in final_model.parameters() if p.requires_grad],
    lr=LR_HEAD_STAGE,
    weight_decay=WEIGHT_DECAY,
)



scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR_HEAD_STAGE,
    steps_per_epoch=len(final_train_head),
    epochs=EPOCHS_HEAD,
    pct_start=0.20,
    div_factor=25.0,
    final_div_factor=1e4,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

for epoch in range(
    1,
    selected_head_epochs + 1,
):
    metrics, _, _ = run_epoch(
        final_model,
        final_train_head,
        train=True,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        frozen_head_stage=True,
    )

    if CHECK_MODEL_FINITE_EACH_EPOCH:
        assert_model_parameters_finite(
            final_model,
            context=f"final head epoch {epoch}",
        )

    print(
        f"FINAL head epoch "
        f"{epoch:02d}/{selected_head_epochs} "
        f"(development horizon {EPOCHS_HEAD})"
    )
    print_epoch("  Train:", metrics)

    final_history.append({
        "phase": "head",
        "epoch": epoch,
        "development_schedule_horizon": EPOCHS_HEAD,
        **{
            f"train_{k}": v
            for k, v in metrics.items()
        },
    })

final_head_state = copy_state_dict(final_model)




if selected_full_epochs > 0:
    final_model.load_state_dict(
        final_head_state,
        strict=True,
    )
    final_model.to(DEVICE)

    for parameter in final_model.backbone.parameters():
        parameter.requires_grad = True

    classifier_module = final_model.backbone.get_classifier()
    classifier_ids = {
        id(parameter)
        for parameter in classifier_module.parameters()
    }

    encoder_parameters = [
        p
        for p in final_model.backbone.parameters()
        if id(p) not in classifier_ids
    ]
    classifier_parameters = list(
        classifier_module.parameters()
    )
    decoder_parameters = list(
        final_model.segmentation_decoder_parameters()
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": encoder_parameters,
                "lr": LR_ENCODER_FULL,
            },
            {
                "params": classifier_parameters,
                "lr": LR_CLASSIFIER_FULL,
            },
            {
                "params": decoder_parameters,
                "lr": LR_DECODER_FULL,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )



    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[
            LR_ENCODER_FULL,
            LR_CLASSIFIER_FULL,
            LR_DECODER_FULL,
        ],
        steps_per_epoch=len(final_train_full),
        epochs=MAX_FULL_EPOCHS,
        pct_start=0.30,
        div_factor=10.0,
        final_div_factor=1000.0,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    for epoch in range(
        1,
        selected_full_epochs + 1,
    ):
        metrics, _, _ = run_epoch(
            final_model,
            final_train_full,
            train=True,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )

        if CHECK_MODEL_FINITE_EACH_EPOCH:
            assert_model_parameters_finite(
                final_model,
                context=f"final full epoch {epoch}",
            )

        print(
            f"FINAL full epoch "
            f"{epoch:02d}/{selected_full_epochs} "
            f"(development horizon {MAX_FULL_EPOCHS})"
        )
        print_epoch("  Train:", metrics)

        final_history.append({
            "phase": "full",
            "epoch": epoch,
            "development_schedule_horizon": (
                MAX_FULL_EPOCHS
            ),
            **{
                f"train_{k}": v
                for k, v in metrics.items()
            },
        })

final_state = copy_state_dict(final_model)

print()
print("=" * 80)
print("FINAL FULL-TRAIN REFIT COMPLETE")
print("=" * 80)
print(
    "Training images:",
    len(official_train_df),
)
print(
    "Training patients:",
    official_train_df["patient_id"].nunique(),
)
print(
    "Selected head-stage epochs:",
    selected_head_epochs,
)
print(
    "Selected full-stage epochs:",
    selected_full_epochs,
)
print(
    "Primary final patient threshold:",
    PRIMARY_PATIENT_THRESHOLD,
)
print(
    "Development-selected secondary threshold:",
    selected_threshold,
)
print("Validation used during final refit: NO")


  0%|          | 0/1193 [00:00<?, ?it/s]

FINAL head epoch 01/3 (development horizon 3)
  Train: loss=0.9686 | cls=0.5721 | seg=0.7930 | Dice=0.6631 | IoU=0.5378 | PredMask=0.095 | GTMask=0.058 | PatientAUC=0.6267 | PatientAUPRC=0.3490 | ImageAUC=0.5906 | Patients=3354


  0%|          | 0/1193 [00:00<?, ?it/s]

FINAL head epoch 02/3 (development horizon 3)
  Train: loss=0.6471 | cls=0.5319 | seg=0.2305 | Dice=0.8293 | IoU=0.7286 | PredMask=0.056 | GTMask=0.057 | PatientAUC=0.7240 | PatientAUPRC=0.4595 | ImageAUC=0.6864 | Patients=3354


  0%|          | 0/1193 [00:00<?, ?it/s]

...

FINAL full epoch 09/11 (development horizon 25)
  Train: loss=0.2820 | cls=0.2219 | seg=0.1203 | Dice=0.9013 | IoU=0.8285 | PredMask=0.057 | GTMask=0.058 | PatientAUC=0.9938 | PatientAUPRC=0.9815 | ImageAUC=0.9844 | Patients=3354


  0%|          | 0/2386 [00:00<?, ?it/s]

FINAL full epoch 10/11 (development horizon 25)
  Train: loss=0.2148 | cls=0.1582 | seg=0.1132 | Dice=0.9070 | IoU=0.8369 | PredMask=0.057 | GTMask=0.057 | PatientAUC=0.9972 | PatientAUPRC=0.9912 | ImageAUC=0.9917 | Patients=3354


  0%|          | 0/2386 [00:00<?, ?it/s]

FINAL full epoch 11/11 (development horizon 25)
  Train: loss=0.1818 | cls=0.1287 | seg=0.1062 | Dice=0.9124 | IoU=0.8450 | PredMask=0.057 | GTMask=0.057 | PatientAUC=0.9980 | PatientAUPRC=0.9959 | ImageAUC=0.9948 | Patients=3354

FINAL FULL-TRAIN REFIT COMPLETE
Training images: 9541
Training patients: 3354
Selected head-stage epochs: 3
Selected full-stage epochs: 11
Primary final patient threshold: 0.5
Development-selected secondary threshold: 0.24900402128696442
Validation used during final refit: NO


In [16]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

final_checkpoint_path = (
    MODEL_DIR
    / f"{FINAL_RUN_NAME}.pt"
)

final_checkpoint = {
    "status": (
        "THYROIDXL_CONVNEXTTINY_FINAL_OFFICIAL_TRAIN_"
        "REFIT_ONEFOLD_SELECTED"
    ),
    "publication_role": (
        "final refit on the complete official training cohort "
        "after one-fold patient-disjoint development selection"
    ),
    "dataset": "ThyroidXL",
    "dataset_repo": REPO_ID,
    "dataset_revision": REPO_REVISION,
    "official_test_accessed": False,
    "internal_validation_used_in_final_refit": False,
    "model_variant": "ConvNeXtTiny_Multiscale_DiceBCE",
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "amp_enabled": bool(USE_AMP),
    "numerical_safety": {
        "fail_on_nonfinite_logits_losses_probabilities": True,
        "check_model_parameters_each_epoch": bool(
            CHECK_MODEL_FINITE_EACH_EPOCH
        ),
    },
    "seed": SEED,
    "drop_rate": DROP_RATE,
    "feature_channels": dict(
        final_model.inferred_feature_channels
    ),
    "parameter_count_total": int(
        sum(p.numel() for p in final_model.parameters())
    ),
    "parameter_count_backbone": int(
        sum(p.numel() for p in final_model.backbone.parameters())
    ),
    "training_images": int(
        len(official_train_df)
    ),
    "training_patients": int(
        official_train_df["patient_id"].nunique()
    ),
    "training_patient_class_counts": {
        str(k): int(v)
        for k, v in patient_counts.items()
    },
    "head_epochs": int(
        selected_head_epochs
    ),
    "full_epochs": int(
        selected_full_epochs
    ),
    "development_head_schedule_horizon": int(
        EPOCHS_HEAD
    ),
    "development_full_schedule_horizon": int(
        MAX_FULL_EPOCHS
    ),
    "selection_source": {
        "split_level": "patient",
        "fold_index": int(FOLD_INDEX),
        "n_folds": int(N_FOLDS),
        "patient_split_seed": int(SEED),
        "training_images": int(
            len(development_train_df)
        ),
        "training_patients": int(
            development_train_df[
                "patient_id"
            ].nunique()
        ),
        "validation_images": int(
            len(development_val_df)
        ),
        "validation_patients": int(
            development_val_df[
                "patient_id"
            ].nunique()
        ),
        "patient_overlap": 0,
        "checkpoint_selection_metric": (
            "validation patient-level ROC-AUC"
        ),
        "best_phase": best_phase,
        "best_epoch": int(best_epoch),
        "best_validation_patient_auc": float(
            best_patient_auc
        ),
        "best_validation_image_auc": float(
            best_val_metrics["image_auc"]
        ),
        "best_validation_segmentation_dice": float(
            best_val_metrics[
                "segmentation_dice"
            ]
        ),
        "best_validation_segmentation_iou": float(
            best_val_metrics[
                "segmentation_iou"
            ]
        ),
    },
    "objective": {
        "classification": "BCEWithLogitsLoss",
        "segmentation": (
            "soft Dice + 0.5 * BCEWithLogitsLoss"
        ),
        "total": (
            "classification BCE + 0.5 * "
            "(soft Dice + 0.5 * segmentation BCE)"
        ),
    },
    "segmentation_loss_weight": (
        SEGMENTATION_LOSS_WEIGHT
    ),
    "segmentation_bce_weight": (
        SEGMENTATION_BCE_WEIGHT
    ),
    "optimizer": "AdamW",
    "weight_decay": WEIGHT_DECAY,
    "learning_rates": {
        "head": LR_HEAD_STAGE,
        "encoder_full": LR_ENCODER_FULL,
        "classifier_full": (
            LR_CLASSIFIER_FULL
        ),
        "decoder_full": LR_DECODER_FULL,
    },
    "batch_sizes": {
        "head": BATCH_HEAD,
        "full": BATCH_FULL,
    },
    "patient_aggregation_primary": (
        "mean malignant probability across frames"
    ),
    "patient_aggregation_benchmark_secondary": (
        "confidence-weighted majority voting"
    ),
    "image_threshold": IMAGE_THRESHOLD,
    "primary_patient_threshold": (
        PRIMARY_PATIENT_THRESHOLD
    ),
    "development_selected_patient_threshold": (
        float(selected_threshold)
    ),

    "validation_patient_threshold": float(
        selected_threshold
    ),
    "development_threshold_source": (
        "Youden J on Fold-1 patient-level "
        "validation mean-probability scores"
    ),
    "threshold_reporting_policy": (
        "report ROC-AUC/AUPRC; report fixed 0.5 "
        "operating point as primary; report development-"
        "selected threshold as pre-specified secondary"
    ),
    "state_dict": final_state,
}

torch.save(
    final_checkpoint,
    final_checkpoint_path,
)

final_checkpoint_sha256 = sha256_file(
    final_checkpoint_path
)

final_history_path = (
    RESULTS_DIR
    / f"{FINAL_RUN_NAME}_training_history.csv"
)
pd.DataFrame(final_history).to_csv(
    final_history_path,
    index=False,
)

final_manifest = {
    key: value
    for key, value in final_checkpoint.items()
    if key != "state_dict"
}

final_manifest.update({
    "checkpoint": str(
        final_checkpoint_path
    ),
    "checkpoint_sha256": (
        final_checkpoint_sha256
    ),
    "development_checkpoint": str(
        development_checkpoint_path
    ),
    "development_checkpoint_sha256": (
        sha256_file(
            development_checkpoint_path
        )
    ),
    "patient_fold_assignments": str(
        fold_assignments_path
    ),
    "development_validation_image_predictions": str(
        validation_image_predictions_path
    ),
    "development_validation_patient_predictions": str(
        validation_patient_predictions_path
    ),
    "versions": {
        "torch": torch.__version__,
        "timm": timm.__version__,
        "albumentations": A.__version__,
    },
    "augmentation": {
        "aspect_ratio_preserved": True,
        "horizontal_flip_p": 0.5,
        "affine_scale": [0.92, 1.06],
        "affine_translate_percent": [
            -0.03,
            0.03,
        ],
        "affine_rotate_degrees": [
            -10,
            10,
        ],
        "affine_shear_degrees": [-2, 2],
        "affine_p": 0.70,
        "brightness_contrast_limit": 0.12,
        "brightness_contrast_p": 0.40,
        "gamma": [85, 115],
        "gamma_p": 0.20,
        "gaussian_blur_p": 0.12,
    },
    "methodological_note": (
        "All images belonging to a patient were kept in the "
        "same development fold. Fold 1 was used only for "
        "development selection. A fresh final model was then "
        "fitted on all 9,541 official-training images. The "
        "official held-out split was not accessed by this "
        "notebook."
    ),
})

final_manifest_path = (
    RESULTS_DIR
    / f"{FINAL_RUN_NAME}_training_manifest.json"
)
final_manifest_path.write_text(
    json.dumps(
        final_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 80)
print("=" * 80)
print(
    "Checkpoint:",
    final_checkpoint_path,
)
print(
    "SHA256:",
    final_checkpoint_sha256,
)
print(
    "History:",
    final_history_path,
)
print(
    "Manifest:",
    final_manifest_path,
)
print()


Checkpoint: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/Models/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_officialtrain9541_onefoldselected_512_seed42_final.pt
SHA256: bd0f34c35f61d6ad3bba3a01758c37dbb71ddbd928ac93949d9bce16bdb8b53f
History: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/results/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_officialtrain9541_onefoldselected_512_seed42_final_training_history.csv
Manifest: /content/drive/MyDrive/ThyroidXL_Publication_OneFold/results/ConvNeXtTiny/convnexttiny_thyroidxl_multiscale_officialtrain9541_onefoldselected_512_seed42_final_training_manifest.json

